# Notebook 3: Summary Memory — `SummarizationNode`
### (the modern version of `ConversationSummaryMemory`)

### What it does (simplest version)
When the conversation gets too long, an LLM **compresses the old turns into a running summary**.
From then on the model sees:  **summary + recent messages** — not the raw full history.

### Human analogy
**Meeting notes.** You don't replay the whole 3-hour meeting in your head;
you read the notes, then follow the live discussion.

### How it works
```
history grows past a limit
        ↓
older messages → LLM → summary (stored in state)
        ↓
next turn: LLM sees [summary] + [recent messages]
```

### Where to use it
Long conversations where old details **matter**: tutoring sessions, coaching bots,
long support tickets. The middle ground — cheaper than full history, forgets less than a window.

### What it connects with later
Notice where the summary lives: **in the agent's state** — so the *checkpointer* saves it too
(Notebook 1's piece, doing double duty). And the idea of *"keep important facts somewhere
separate"* is the seed of long-term memory → Notebooks 4–6.

## Step 1 — Setup

In [ ]:
from typing import TypedDict
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.messages import AnyMessage
from langchain_core.messages.utils import count_tokens_approximately
from langgraph.graph import StateGraph, MessagesState, START
from langgraph.checkpoint.memory import InMemorySaver
from langmem.short_term import SummarizationNode, RunningSummary

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("Setup done.")

## Step 2 — The graph: summarize first, then answer

Two new ideas here:

1. **`State` gets an extra field `context`** — that's where the running summary lives.
2. **Two nodes in a row:** `summarize` (compresses old messages if needed) → `call_model`
   (answers using the summarized messages). Summarize runs **first**, every turn.

In [ ]:
class State(MessagesState):
    context: dict[str, RunningSummary]   # <- the running summary lives here

class LLMInputState(TypedDict):          # what the model node receives
    summarized_messages: list[AnyMessage]
    context: dict[str, RunningSummary]

# This node summarizes old messages once the history passes the limit
summarization_node = SummarizationNode(
    token_counter=count_tokens_approximately,
    model=llm.bind(max_tokens=256),
    max_tokens=512,
    max_tokens_before_summary=512,   # summarize when history exceeds this
    max_summary_tokens=256,
)

def call_model(state: LLMInputState):
    response = llm.invoke(state["summarized_messages"])
    return {"messages": [response]}

builder = StateGraph(State)
builder.add_node(call_model)
builder.add_node("summarize", summarization_node)
builder.add_edge(START, "summarize")         # summarize FIRST...
builder.add_edge("summarize", "call_model")  # ...then answer
graph = builder.compile(checkpointer=InMemorySaver())
print("Agent ready with summary memory.")

## Step 3 — Have a long conversation

We share two important facts early (name + goal), then chat about many topics
so the history grows past the summary limit.

In [ ]:
config = {"configurable": {"thread_id": "summary-demo"}}

graph.invoke({"messages": "Hi, my name is Rahul."}, config)
graph.invoke({"messages": "I'm preparing for a LangChain teaching job."}, config)
print("Early facts shared: name + goal")

for topic in ["prompt templates", "chains", "agents", "retrieval", "memory"]:
    graph.invoke({"messages": f"Explain {topic} briefly in two paragraphs."}, config)
    print(f"...covered {topic}")

## Step 4 — The test: did the early facts survive?

In Notebook 2 (window), the name was **gone** by this point.
With a summary, it should still be there — compressed, not deleted.

In [ ]:
final = graph.invoke(
    {"messages": "What is my name, and what am I preparing for?"}, config
)
print("Final answer:")
final["messages"][-1].pretty_print()

In [ ]:
# Now the cool part — READ the agent's memory directly:
print("=== Running summary stored in state ===")
print(final["context"]["running_summary"].summary)
# The early facts survived inside this compressed text.

## Try it yourself

1. Print `len(final["messages"])` — why is it much shorter than the full raw history?
   (The summarized messages are **replaced** in state, keeping things compact.)
2. Lower `max_tokens_before_summary` to 200 — summarization kicks in earlier. Watch the
   summary appear after fewer turns.
3. Compare all three so far, honestly, as a table:

| Approach | Cost per turn | Remembers old details? | Old LangChain twin |
|---|---|---|---|
| Full history (NB1) | grows forever | yes, everything | `ConversationBufferMemory` |
| Window (NB2) | fixed, cheap | no — drops them | `ConversationBufferWindowMemory` |
| Summary (NB3) | moderate | yes — compressed | `ConversationSummaryMemory` |

**Next notebook:** all three die with the `thread_id`. Time to remember users
*across* conversations → the **Store**.